# Autonomous Movie Studio — GPU Validation (Colab)

This notebook performs end-to-end GPU validation of the pipeline.

**Before running:**
1. Go to Runtime → Change runtime type → select GPU (T4, L4, A100, etc.)
2. Run cells in order.


In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install system packages
!apt-get update -y && apt-get install -y ffmpeg git

In [ ]:
# Upgrade pip
!python -m pip install --upgrade pip setuptools wheel

In [ ]:
# Install PyTorch with CUDA support (auto-detect CUDA version in Colab)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# Install WhisperX and dependencies
!pip install git+https://github.com/m-bain/whisperX.git

In [ ]:
# Install PySceneDetect and OpenCV
!pip install scenedetect[opencv] opencv-python-headless

In [ ]:
# Optional: pyttsx3 for fixture synthesis (some Colab runtimes have issues with pyttsx3)
!pip install pyttsx3 || echo 'pyttsx3 install optional; will fall back to silent fixture'

In [ ]:
# Clone repository
# Replace REPO_URL with your fork or the main repo
REPO_URL = 'https://github.com/asdfhgds/automovies.git'
!git clone {REPO_URL} repo
%cd repo

In [ ]:
# Run doctor to confirm environment
!python src/main.py doctor

In [ ]:
# Generate tiny test fixture with speech
!python tests/fixtures/generate_test_fixture.py tests/fixtures/test_speech.mp4 "This is a short GPU validation test."

In [ ]:
# Verify fixture exists and check duration
!ls -lh tests/fixtures/test_speech.mp4
!ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 tests/fixtures/test_speech.mp4

In [ ]:
# Run the normal test suite (fast tests only)
import os
os.environ['PYTHONPATH'] = 'src'
!pytest -q -m "not integration"

In [ ]:
# Init project with test fixture
!python src/main.py init --title "Colab GPU Validation" --source "$(pwd)/tests/fixtures/test_speech.mp4"

In [ ]:
# Extract project ID from the previous output
# (You may need to copy/paste the project ID from the init output above)
# For automation, we can also extract it:
import json
from pathlib import Path

data_dir = Path('data')
projects = sorted(data_dir.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
if projects:
    PROJECT_ID = projects[0].name
    print(f"Using project: {PROJECT_ID}")
else:
    print("No project found")

In [ ]:
# Run the pipeline
import subprocess
import os
os.environ['PYTHONPATH'] = 'src'
result = subprocess.run(['python', 'src/main.py', 'run', '--project-id', PROJECT_ID], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)

In [ ]:
# List generated artifacts
!echo "=== Transcripts ===" && ls -lh data/$PROJECT_ID/transcripts/ || echo "No transcripts"
!echo "=== Scenes ===" && ls -lh data/$PROJECT_ID/scenes/ || echo "No scenes"
!echo "=== Assets/Scenes ===" && ls -lh data/$PROJECT_ID/assets/scenes/ || echo "No extracted scenes"
!echo "=== Director Plan ===" && test -f data/$PROJECT_ID/director_plan.json && echo "FOUND" || echo "NOT FOUND"

In [ ]:
# Inspect transcript.json
import json
transcript_path = Path(f'data/{PROJECT_ID}/transcripts/transcript.json')
if transcript_path.exists():
    with open(transcript_path) as f:
        transcript = json.load(f)
    print("=== Transcript Summary ===")
    print(f"Provider: {transcript.get('provider')}")
    print(f"Source: {transcript.get('source')}")
    print(f"Language: {transcript.get('language')}")
    print(f"Segments: {len(transcript.get('segments', []))}")
    if transcript.get('segments'):
        seg = transcript['segments'][0]
        print(f"First segment: {seg}")
else:
    print("No transcript.json")

In [ ]:
# Inspect scene_index.json
scene_index_path = Path(f'data/{PROJECT_ID}/scenes/scene_index.json')
if scene_index_path.exists():
    with open(scene_index_path) as f:
        scenes = json.load(f)
    print("=== Scene Index ===")
    print(f"Total scenes: {len(scenes)}")
    for i, scene in enumerate(scenes[:3]):
        print(f"\nScene {i}: {scene.get('scene_id')}")
        print(f"  Start: {scene.get('start_sec')} End: {scene.get('end_sec')}")
        print(f"  Transcript length: {len(scene.get('transcript', ''))}")
else:
    print("No scene_index.json")

In [ ]:
# Inspect selected_scene.json
selected_path = Path(f'data/{PROJECT_ID}/scenes/selected_scene.json')
if selected_path.exists():
    with open(selected_path) as f:
        selected = json.load(f)
    print("=== Selected Scene ===")
    print(json.dumps(selected, indent=2))
else:
    print("No selected_scene.json")

In [ ]:
# Probe the extracted clip
import subprocess
clip_dir = Path(f'data/{PROJECT_ID}/assets/scenes')
if clip_dir.exists():
    clips = list(clip_dir.glob('*.mp4'))
    if clips:
        clip = clips[0]
        print(f"=== Extracted Clip ===")
        print(f"Path: {clip}")
        print(f"Size: {clip.stat().st_size / 1024 / 1024:.2f} MB")
        # ffprobe for duration and properties
        res = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', str(clip)], capture_output=True, text=True)
        if res.returncode == 0:
            dur = float(res.stdout.strip())
            print(f"Duration: {dur:.2f} seconds")
    else:
        print("No clips found")
else:
    print("Assets/scenes directory not found")

In [ ]:
# Run integration test
os.environ['PYTHONPATH'] = 'src'
result = subprocess.run(['pytest', '-m', 'integration', '-v'], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)
print(f"Return code: {result.returncode}")

In [ ]:
# Final summary
print("\n" + "="*60)
print("GPU VALIDATION SUMMARY")
print("="*60)
print(f"Project ID: {PROJECT_ID}")
print(f"\nArtifacts produced:")
for artifact in [
    'transcripts/transcript.json',
    'scenes/scene_index.json',
    'scenes/scene_ranking.json',
    'scenes/selected_scene.json',
    'director_plan.json',
]:
    path = Path(f'data/{PROJECT_ID}/{artifact}')
    status = "✓" if path.exists() else "✗"
    print(f"  {status} {artifact}")

print(f"\nExtracted clip:")
clips = list(Path(f'data/{PROJECT_ID}/assets/scenes').glob('*.mp4'))
if clips:
    print(f"  ✓ {clips[0].name}")
else:
    print(f"  ✗ No clips found")

print("\n" + "="*60)
print("To update PROJECT_STATUS.md, note:")
print(f"- Project ID: {PROJECT_ID}")
print(f"- Run command: python src/main.py run --project-id {PROJECT_ID}")
print("="*60)